In [50]:
# Import Libraries

import kagglehub
import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Download nltk data
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\buddh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\buddh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### Step-1. Data Understanding

In [51]:
# Download dataset
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

# Load CSV
df = pd.read_csv(path + "/IMDB Dataset.csv")

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [52]:
print("Dataset shape:", df.shape)

# Class distribution
print(df['sentiment'].value_counts())

# Sample data
df.sample(5)

Dataset shape: (50000, 2)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


,review,sentiment
20250,A well written screenplay. A moving story show...,positive
29832,It's said that this film is or was banned in t...,positive
31121,It was very heart-warming. As an expatriated f...,positive
7421,This really is a great film. Full of love and ...,positive
48552,This film has the worst editing I've ever seen...,negative


### Steo-2. NLP Preprocessing

In [53]:
# Create reusable function

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Tokenization
    words = text.split()
    
    # Remove stopwords & lemmatize
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    return " ".join(words)

In [54]:
# Apply preprocessing

df['clean_text'] = df['review'].apply(preprocess_text)

df.head()

,review,sentiment,clean_text
0,One of the other reviewers has mentioned that ...,positive,one reviewer mentioned watching oz episode you...
1,A wonderful little production. <br /><br />The...,positive,wonderful little production br br filming tech...
2,I thought this was a wonderful way to spend ti...,positive,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,negative,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love time money visually stunni...


### Step-3. Feature Engineering 

In [56]:
# Train-Test Split

X = df['clean_text']
y = df['sentiment']

# Convert labels to 0/1
y = y.map({'positive': 1, 'negative': 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [57]:
#Bag of Words

bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

In [58]:
# TF-IDF

tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

### Step-4. Model Building

In [64]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    return acc, prec, rec, f1


def print_metrics(name, acc, prec, rec, f1):
    print(f"{name}")
    print("Accuracy :", round(acc, 4))
    print("Precision:", round(prec, 4))
    print("Recall   :", round(rec, 4))
    print("F1 Score :", round(f1, 4))
    print("-" * 40)

In [65]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier()
}

In [66]:
results_list = []

for model_name, model in models.items():
    
    # -------- BoW --------
    model.fit(X_train_bow, y_train)
    acc, prec, rec, f1 = evaluate_model(model, X_test_bow, y_test)
    
    print_metrics(f"{model_name} (BoW)", acc, prec, rec, f1)
    
    results_list.append([model_name + " BoW", acc, prec, rec, f1])
    
    
    # -------- TF-IDF --------
    model.fit(X_train_tfidf, y_train)
    acc, prec, rec, f1 = evaluate_model(model, X_test_tfidf, y_test)
    
    print_metrics(f"{model_name} (TF-IDF)", acc, prec, rec, f1)
    
    results_list.append([model_name + " TF-IDF", acc, prec, rec, f1])

Logistic Regression (BoW)
Accuracy : 0.8727
Precision: 0.8688
Recall   : 0.8803
F1 Score : 0.8745
----------------------------------------
Logistic Regression (TF-IDF)
Accuracy : 0.8846
Precision: 0.8765
Recall   : 0.8974
F1 Score : 0.8868
----------------------------------------
Naive Bayes (BoW)
Accuracy : 0.8457
Precision: 0.8497
Recall   : 0.8428
F1 Score : 0.8463
----------------------------------------
Naive Bayes (TF-IDF)
Accuracy : 0.8519
Precision: 0.8512
Recall   : 0.8557
F1 Score : 0.8534
----------------------------------------
Decision Tree (BoW)
Accuracy : 0.714
Precision: 0.7187
Recall   : 0.7105
F1 Score : 0.7146
----------------------------------------
Decision Tree (TF-IDF)
Accuracy : 0.716
Precision: 0.7213
Recall   : 0.7113
F1 Score : 0.7162
----------------------------------------


### Step-5. Model Evaluation 

In [67]:
results = pd.DataFrame(results_list, columns=[
    'Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score'
])

results

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression BoW,0.8727,0.868782,0.880333,0.874519
1,Logistic Regression TF-IDF,0.8846,0.876526,0.897400,0.886841
2,Naive Bayes BoW,0.8457,0.849740,0.842826,0.846269
3,Naive Bayes TF-IDF,0.8519,0.851165,0.855725,0.853439
4,Decision Tree BoW,0.7140,0.718731,0.710458,0.714571
5,Decision Tree TF-IDF,0.7160,0.721272,0.711252,0.716227


### Step-6. Comparison & Insights

#### 🔹 1. Best Preprocessing Steps
Lowercasing helped normalize text and reduce duplicate words
Removing URLs, punctuation, and special characters removed noise
Stopword removal reduced non-informative words
Lemmatization improved word consistency (e.g., “running” → “run”)

#### 🔹 2. Best Vectorization Technique
TF-IDF performed better than Bag of Words (BoW) across most models

Reason:

TF-IDF reduces importance of frequently occurring words
Highlights rare but important sentiment words
Leads to better generalization

#### 🔹 3. Best Model
Logistic Regression → Best overall performance (high accuracy + stable F1)
Naive Bayes → Strong precision but weaker recall
Decision Tree → Balanced recall but lower accuracy

#### 🔹 4. Trade-offs Between Models

✅ Logistic Regression

Pros:

High accuracy and F1 score
Works very well with TF-IDF
Stable and reliable

Cons:

Slightly lower recall in some cases
Assumes linear relationship (may miss complex patterns)

✅ Naive Bayes

Pros:

Very fast and efficient
Performs well on text data
High precision

Cons:

Lower recall → misses some positive/negative cases
Assumes feature independence (not always true in text)

✅ Decision Tree

Pros:

Easy to interpret
Captures non-linear relationships
Better recall in some cases

Cons:

Lower accuracy compared to LR and NB
Prone to overfitting
Less stable